In [1]:
import numpy as np 
import pandas as pd
from scipy import stats
from numpy import random
from pathlib import Path
from statsmodels.distributions.copula.api import GumbelCopula

In [2]:
# utility functions

def make_pos_def(corr):
    eigvals, eigvecs = np.linalg.eigh(corr)
    eigvals[eigvals < 1e-8] = 1e-8  
    corr_pd = eigvecs @ np.diag(eigvals) @ eigvecs.T

    # normalize diagonal to 1
    d = np.sqrt(np.diag(corr_pd))
    corr_pd = corr_pd / d[:, None] / d[None, :]
    return corr_pd

def beta_GC(R, n, a,b,rng=None):
    R_pos = make_pos_def(R)
    mean = np.zeros(np.shape(R_pos)[0])
    if rng is None:
        z = np.random.multivariate_normal(mean, R_pos, size=n)
    else:
        z = rng.multivariate_normal(mean, R_pos, size=n)
    u = stats.norm.cdf(z)
    data = np.zeros_like(u)
    for i in range(u.shape[1]):
        data[:, i] = stats.beta.ppf(u[:, i], a[i], b[i])
    return data, R_pos

# def beta_GC_nonlinear(n, a, b, theta, nobj, rng):
#     copula = GumbelCopula(theta=theta, k_dim=nobj+1)
#     u = copula.rvs(nobs=n, random_state=rng)
#     data = np.zeros_like(u)
#     for i in range(u.shape[1]):
#         data[:, i] = stats.beta.ppf(u[:, i], a[i], b[i])
#     return data

def cleanupsamples(samples, nobj, precision=1):
    """Clean up samples by rounding and removing duplicates."""
    samples = np.round(samples, precision)
    c, i = np.unique(samples[:, :nobj], axis=0, return_index=True)
    newsamples = samples[i, :]  # note - these have been sorted into increasing magnitude
    if precision == 0:
        newsamples = np.array(newsamples, dtype=np.int32)
    return newsamples

def generate_beta_example_data(r, a, b, max_val, nobj, n_items=100, seed=1124, precision=0):
    item_rng = random.default_rng(seed=seed)
    
    batch = max(5, n_items // 10)
    uniq = set()
    items = []
    while len(items) < n_items:
        new, rpos = beta_GC(r, batch, a, b, rng=item_rng)
        new = 1 + new * (max_val-2)
        new = cleanupsamples(new, nobj=nobj, precision=precision)
        for item in new:
            key = tuple(item[:nobj])
            if key not in uniq:
                uniq.add(key)
                items.append(item)
                if len(items) == n_items:
                    break
    return np.unique(np.array(items), axis=0), rpos # here np.unique is used for sorting

# def generate_beta_example_data_nonlinear(theta, a, b, max_val, nobj, n_items=100, seed=1124, precision=0):
#     item_rng = random.default_rng(seed=seed)
    
#     batch = max(5, n_items // 10)
#     uniq = set()
#     items = []
#     while len(items) < n_items:
#         new = beta_GC_nonlinear(batch, a, b, theta, nobj, item_rng)
#         new = 1 + new * (max_val-2)
#         new = cleanupsamples(new, nobj=nobj, precision=precision)
#         for item in new:
#             key = tuple(item[:nobj])
#             if key not in uniq:
#                 uniq.add(key)
#                 items.append(item)
#                 if len(items) == n_items:
#                     break
#     return np.unique(np.array(items), axis=0)

In [3]:
n_obj = 3
n_items = 60
items_seed = 1125
objective_range = 20
# a = [1.5,1.2,1.3,1.6,1.4,1]
# b = [2.1,1.8,1.6,1.9,2.0,1]
a = [1.5,1.3,1.6,1]
b = [2.1,1.6,1.9,1]
r = -0.3 * np.ones((n_obj + 1, n_obj + 1)) + (1 + 0.3) * np.eye(n_obj + 1)
print("objective value corr:\n", r)
cost_corr = 0.3
r[:, n_obj] = cost_corr
r[n_obj, :] = cost_corr
r[n_obj, n_obj] = 1
print("full corr:\n",r)

objective value corr:
 [[ 1.  -0.3 -0.3 -0.3]
 [-0.3  1.  -0.3 -0.3]
 [-0.3 -0.3  1.  -0.3]
 [-0.3 -0.3 -0.3  1. ]]
full corr:
 [[ 1.  -0.3 -0.3  0.3]
 [-0.3  1.  -0.3  0.3]
 [-0.3 -0.3  1.   0.3]
 [ 0.3  0.3  0.3  1. ]]


In [4]:
items, rpos = generate_beta_example_data(r, a, b, max_val=objective_range, nobj=n_obj, n_items=n_items, seed=items_seed)
itemsdf = pd.DataFrame(items)
output_path = Path("data/items")
itemsdf.to_csv(output_path / f'items_obj{n_obj}_seed{items_seed}.csv', index=False, header=False)

In [ ]:
print("positive definite corr:\n", rpos)

In [ ]:
print("items corr:\n", np.round(np.corrcoef(items.T), 2))

In [ ]:
# # generate nonlinear betas

# n_obj = 5
# n_items = 60
# items_seed = 1125
# objective_range = 20
# a = [1.5,1.2,1.3,1.6,1.4,1]
# b = [2.1,1.8,1.6,1.9,2.0,1]
# theta = 2.0

# items = generate_beta_example_data_nonlinear(theta, a, b, objective_range, n_obj, n_items=n_items, seed=items_seed)
# itemsdf = pd.DataFrame(items)
# output_path = Path("data/items")
# itemsdf.to_csv(output_path / f'items_nonlinear_obj{n_obj}_seed{items_seed}.csv', index=False, header=False)